In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
plt.rcParams["animation.html"]="jshtml"

List_methode_de_flux= ["Roe","Rusanov","Godunov"]
List_condition_limite=["Neumann","Dirichlet","Periodique"]

In [ ]:
class Finite_Volume:


    def __init__(self,u0,function, derivee_de_la_function, flux_methode, film_bool =False, condition_limite = "Neumann",  ghost_nodes=[0,0], domaine=(0.,1.) ):
        Valid_init = True
        self.function = function; self.derivee_de_la_function = derivee_de_la_function; self.flux = lambda u,v:0*u+0*v

        in_list =False
        for methode in List_methode_de_flux:
            if(flux_methode ==methode):  in_list =True
        if(not in_list): Valid_init = False

        in_list =False
        for condition in List_condition_limite:
            if(condition_limite ==condition): in_list =True
        if(not in_list): Valid_init = False
        
    
        self.flux_methode = flux_methode;   self.condition_limite = condition_limite
        self.ghost_nodes=ghost_nodes
        self.u0=u0
        self.film_bool = film_bool
        self.domaine= domaine
        if(not Valid_init): f"Methode non reconnue"
        
    def g_godunov(self, u,v):
        U = np.linspace(u, v, 10)
        if(u<v):    return np.min(self.function(U))
        else :      return np.max(self.function(U))

    def flux_function(self,problem):
        
        if(self.flux_methode == "Godunov"): 
            if(problem =="Burgers"): self.flux= lambda u,v :  (0 if (u*v<0 and self.function(1)>0)  else min(self.function(u),self.function(v))) if u<v else (0 if(u*v<0 and self.function(1)<0) else max(self.function(u),self.function(v)))
            elif(problem =="cubique"): self.flux= lambda u,v : min(self.function(u),self.function(v)) if u<v else max(self.function(u),self.function(v))
            else : 
                self.flux= lambda u,v : self.g_godunov(u,v)
        elif(self.flux_methode == "Rusanov"):
            self.flux= lambda u,v : 0.5*(self.function(u)+self.function(v))-0.5*max(np.abs(self.derivee_de_la_function(u)), np.abs(self.derivee_de_la_function(v)))*(v-u)
        elif(self.flux_methode == "Roe"):
            self.flux= lambda u,v : (0.5*(self.function(u)+self.function(v))-0.5*np.abs(self.derivee_de_la_function(u))*(v-u))*(u==v) + (self.function(u)-self.function(v))*(v<u)

    def sol(self, J,T,  CFL=0.5):
        dx = 1./J
        X = np.linspace(self.domaine[0]+dx,self.domaine[1],J) -dx/2

        f = self.function; f_p = self.derivee_de_la_function; flux = self.flux
        U = self.u0(X)

        FG = np.empty_like(X) 
        FD = np.empty_like(X)

        
        U_anime = []; dt_list=[]

        t=0.; n =0; dt=0.
        while t<T and n<10000:
            #calcul de dt
            vitesse = max(abs(f_p(U)))
            if(vitesse !=0) : dt = (min(CFL * dx /(2* vitesse), T-t))
            else :            dt = CFL*dx  

            if(dt<0): dt = 10e-10; print(f"!!!!!!!!!!!!!!!!! dt<0 !!!!!!!!!")

            ## Calcul du flux
            for j in range(len(X)):
                
                if(j==0):   
                    if(self.condition_limite =="Dirichlet"):    FG[j] = flux(self.ghost_nodes[0],       U[j])
                    elif(self.condition_limite =="Neumann"):    FG[j] = flux(U[j]+self.ghost_nodes[0],U[j])
                    elif(self.condition_limite =="Periodique"): FG[j] = flux(U[-1], U[j])

                else :      FG[j]= flux(U[j-1],U[j]); 
                
                if(j==J-1):   
                    if(self.condition_limite =="Dirichlet"):    FD[j] = flux(U[j], self.ghost_nodes[1])
                    elif(self.condition_limite =="Neumann"):    FD[j] = flux(U[j], U[j]+self.ghost_nodes[1])
                    elif(self.condition_limite =="Periodique"): FD[j] = flux(U[j], U[0])

                else :      FD[j]= flux(U[j],U[j+1]) 
               
            ## eval
            if(n%10==0):
                U_anime.append(U.copy())

            ## Calcul de la solution
            for j in range(len(X)):
                U[j] = U[j] -(dt/dx)* (FD[j]-FG[j])

            n+=1; t+= dt; dt_list.append(dt)

        self.anime = U_anime; self.dt_list = dt_list

    def show_sol(self):

        lenght_animation = len(self.anime); T= sum(self.dt_list); L=self.domaine[1]-self.domaine[0]
        print(lenght_animation)

        J = len(self.anime[0]); 
        dx = (self.domaine[1]- self.domaine[0])/J

        X= np.linspace(self.domaine[0],self.domaine[1],J,endpoint=False) -dx/2

        fig, (ax , ax_film) = plt.subplots(1,2)
        
        imag_xt =  np.zeros((lenght_animation,J))
        for i in range(lenght_animation) : imag_xt[i,:] = self.anime[lenght_animation -i-1]
        im = ax_film.imshow(imag_xt,extent=((self.domaine[0], self.domaine[1], 0,T)))
        

        fig.colorbar(im, ax=ax_film, label='Interactive colorbar')
        ax_film.set_aspect(L/T)
        ax_film.set_ylabel("t")
        ax_film.set_xlabel("x")

        line = ax_film.plot(X, 0*X,'r')[0]

        m = np.min(self.anime); M = np.max(self.anime)

        ax.set(xlim=[self.domaine[0],self.domaine[1]], ylim=[m-1,M+1], xlabel='x', ylabel='U')

        u_plot = ax.plot(X, self.anime[0],'b', label="u")[0]

        ax.legend(fancybox=True, shadow=True, loc='upper left')     


        def update(i):
            t= sum(self.dt_list[:i])
            print(i+1,"/",lenght_animation)
            # IPython.display.clear_output(wait=True)
            u_plot.set_ydata(self.anime[i])
            line.set_ydata(i*T/(lenght_animation) +0*X)
            

        anime = animation.FuncAnimation(fig=fig, func=update, frames=lenght_animation)
        return anime
        

In [ ]:
#
ul=-1.;ur=2.
U0  = lambda x: ul*(x<0.5) + ur*(x>=0.5)
f  = lambda u: -0.5* u**2
f_p= lambda u: -u

Fv = Finite_Volume(U0,function=f,derivee_de_la_function=f_p,flux_methode="Godunov",film_bool=True, condition_limite="Neumann")
Fv.flux_function("Burgers")
Fv.sol(J=200,T=0.5,CFL=1.5)

anime =Fv.show_sol()
plt.close()
anime

In [ ]:
ul=-1.;ur=2.
U0  = lambda x: ul*(x<0.5) + ur*(x>=0.5)
f  = lambda u: 0.5* u**2
f_p= lambda u: u

Fv = Finite_Volume(U0,function=f,derivee_de_la_function=f_p,flux_methode="Godunov",film_bool=True, condition_limite="Neumann")
Fv.flux_function("Burgers")
Fv.sol(J=200,T=0.5,CFL=0.95)

anime =Fv.show_sol()
plt.close()
anime

In [ ]:
#
U0  = lambda x: np.sin(2*np.pi*x)
f  = lambda u: 0.5* u**2
f_p= lambda u: u

Fv = Finite_Volume(U0,function=f,derivee_de_la_function=f_p,flux_methode="Godunov",film_bool=True, condition_limite="Neumann")
Fv.flux_function("Burgers")
Fv.sol(J=200,T=0.5,CFL=0.95)

anime =Fv.show_sol()
plt.close()
anime

In [ ]:
#
ul=2.;ur=-2.
U0  = lambda x: ul*(x<0.5) + ur*(x>=0.5)
f  = lambda u: u**3
f_p= lambda u: 3*u**2

Fv = Finite_Volume(U0,function=f,derivee_de_la_function=f_p,flux_methode="Godunov",film_bool=True, condition_limite="Neumann")
Fv.flux_function("Cubique")
Fv.sol(J=200,T=0.5,CFL=0.95)

anime =Fv.show_sol()
plt.close()
anime

In [ ]:
#
ul=-2.;ur=2.
U0  = lambda x: ul*(x<0.5) + ur*(x>=0.5)
f  = lambda u: u**3
f_p= lambda u: 3*u**2

Fv = Finite_Volume(U0,function=f,derivee_de_la_function=f_p,flux_methode="Godunov",film_bool=True, condition_limite="Neumann")
Fv.flux_function("Cubique")
Fv.sol(J=100,T=0.2,CFL=0.95)

anime =Fv.show_sol()
plt.close()
anime